## Prepare Dataset

In [10]:
import pandas as pd
import json


# Dental dataset
#from datasets import load_dataset
#ds = load_dataset("Lines/Open-Domain-Oral-Disease-QA-Dataset")

# Load dataset
df = pd.read_excel("/Users/sgrebenkin/programming/json/70-fixed-batch-inference.xlsx", sheet_name="review")
print('Original dataset length', len(df))

# Filter dataset
df = df.drop(df[df['reference_output_69'].str.find('PARSE_FAILED') != -1].index)
print('Filtered dataset length', len(df))

df['output'] = df.apply(lambda x: json.loads(x['reference_output_69']), axis=1)
df['input'] = df['raw_input'].copy()
print("Max input length", df['input'].str.len().max())

c = {}
for index, row in df.iterrows():
    x = row['output']
    for key, value in x.items():
        c[key] = c.get(key, 0) + int(len(value) > 0)

categories = sorted(list(c.keys()))

# Validation of the dataset
for index, row in df.iterrows():
    a=[0] * len(categories)
    try:
        x = json.loads(row['reference_output_69'])
        if type(x) is dict:
            # Check dental features for ex.
            if 'dental_features' in x and len(x['dental_features'])>0:
                a.append(1)
            else:
                a.append(0)
        else:
            print(x['answer'])
    except ValueError:  # includes simplejson.decoder.JSONDecodeError
        print('Decoding JSON has failed')

# print(df['output'].loc[df.index[2]])
for i in categories:
    print(c[i], '\t', i)
df['output_vector'] = df.apply(lambda x: [int(len(x['output'].get(i, [0])) > 0) for i in categories], axis=1)
df['output_vector']

Original dataset length 5492
Filtered dataset length 5451
Max input length 2078
2141 	 alignment
395 	 as_previous
1197 	 bite
181 	 crowding
1375 	 dental_features
472 	 finishing
485 	 leveling
272 	 midline
95 	 non_clinical_reason_for_new_order
1174 	 occlusion
1833 	 other_instructions
484 	 overcorrection_aligners
689 	 passive_aligners
1630 	 polite_expressions
55 	 request_for_clin_check
199 	 skip_active_treatment
1272 	 spaces
2648 	 teeth_movements
124 	 tracking
394 	 treatment_length


0       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...
1       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2       [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, ...
3       [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...
4       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                              ...                        
5487    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
5488    [1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...
5489    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...
5490    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
5491    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: output_vector, Length: 5451, dtype: object

## Wrap the dataset into the class

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Prepare the dataset
class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=3072):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            #return_attention_mask=True,
            return_attention_mask=False,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            #'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Float for multi-label
        }

# Prepare data for training
X_train, X_val, y_train, y_val = train_test_split(df['input'], df['output_vector'], test_size=0.2)

In [3]:
k = 0
print(X_train[0], y_train[0])
for i in range(len(categories)):
    print(categories[i], y_train[0][i])

[NumberingSystem]FDI

[FormInstructionsLowerArch:] Dear technician, please: replace the intraoral photos by scan;

- The correction of tooth 3.8 should be performed in the sequence:
- mesial in rotation
- intrusion
- Upright through axial torque from crown to distal (70%) and root to mesial (30%)
- Finally, uncrossing, through Lingual crown torque and body movement;
- All movements with speed reduced by 30%;

thankful

-

 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
alignment 0
as_previous 0
bite 0
crowding 0
dental_features 0
finishing 0
leveling 0
midline 0
non_clinical_reason_for_new_order 0
occlusion 0
other_instructions 1
overcorrection_aligners 0
passive_aligners 0
polite_expressions 1
request_for_clin_check 0
skip_active_treatment 0
spaces 0
teeth_movements 1
tracking 0
treatment_length 0


In [4]:
from tqdm import trange, tqdm

# Load the tokenizer and model
model_name = 'roberta-base'  # Change as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(categories))

def configure_training(model):
    # Freeze all existing parameters
    for param in model.parameters():
        param.requires_grad = False
    
    # Last layer unblock
    #for param in model.roberta.encoder.layer[-1].parameters():
    #     param.requires_grad = True
    
    # Unfreeze the classification head
    for param in model.classifier.parameters():
        param.requires_grad = True
    
    # Print out which layers are trainable for verification
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"Trainable layer: {name}")

print(model)

/Users/sgrebenkin/miniconda3/envs/cs224u/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
# Prepare data for training
X_train, X_val, y_train, y_val = train_test_split(df['input'].tolist(), df['output_vector'].tolist(), test_size=0.2)

# Create datasets
train_dataset = CustomDataset(X_train, y_train, tokenizer)
val_dataset = CustomDataset(X_val, y_val, tokenizer)

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)


# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids']
            #attention_mask = batch['attention_mask']
            labels = batch['labels']

            #outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            outputs = model(input_ids=input_ids)
            preds = torch.sigmoid(outputs.logits).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    predictions = [(p > 0.5).astype(int) for p in predictions]
    accuracy = accuracy_score(true_labels, predictions)
    print(f'Validation Accuracy: {accuracy:.4f}')


# Training function
def train_model(model, train_loader, val_loader, epochs=3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    #model.train()
    for param in model.parameters():
        param.requires_grad = False

    for param in model.classifier.parameters():
        param.requires_grad = True
    
    for epoch in range(epochs):
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            input_ids = batch['input_ids']
            #attention_mask = batch['attention_mask']
            labels = batch['labels']
            
            #outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
        
        print(f'Epoch {epoch + 1}/{epochs} completed. Loss: {loss.item()}')
    
    evaluate_model(model, val_loader)

In [7]:
# Train the model
train_model(model, train_loader, val_loader, epochs=10)

# Save the model
model.save_pretrained('roberta_multiclass_model2')
tokenizer.save_pretrained('roberta_multiclass_model2')

100%|█████████████████████████████████████████████████████████████████████████████████| 2180/2180 [02:36<00:00, 13.91it/s]


Epoch 1/10 completed. Loss: 0.24761231243610382


100%|█████████████████████████████████████████████████████████████████████████████████| 2180/2180 [02:38<00:00, 13.78it/s]


Epoch 2/10 completed. Loss: 0.5183922052383423


100%|█████████████████████████████████████████████████████████████████████████████████| 2180/2180 [02:42<00:00, 13.42it/s]


Epoch 3/10 completed. Loss: 0.4696422517299652


100%|█████████████████████████████████████████████████████████████████████████████████| 2180/2180 [02:54<00:00, 12.48it/s]


Epoch 4/10 completed. Loss: 0.3983069360256195


 11%|████████▉                                                                         | 239/2180 [00:18<02:26, 13.27it/s]


KeyboardInterrupt: 